# Clasificación de noticias mediante procesamiento de lenguaje natural

## Objetivo
Desarrollar y comparar varios modelos supervisados para clasificar noticias en categorías temáticas usando técnicas de PLN y representación vectorial del texto.

## Entregables esperados
- Exploración del dataset y balance de clases
- Preprocesamiento del texto
- Vectorización y entrenamiento de al menos tres modelos
- Validación cruzada y métricas
- Análisis de errores y selección del modelo final


## 1. Configuración inicial

Ejecuta esta celda primero. Si tu dataset usa otros nombres de archivo o columnas, ajústalos aquí.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

BASE_DIR = Path.cwd()
FIGURES_DIR = BASE_DIR / "figuras"
FIGURES_DIR.mkdir(exist_ok=True)

TRAIN_PATH = BASE_DIR / "train.csv"
TEST_PATH = BASE_DIR / "test.csv"

TRAIN_PATH, TEST_PATH


## 2. Descarga de recursos de NLTK

Ejecuta esta celda una sola vez si aún no tienes descargados estos recursos.


In [ ]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")


## 3. Carga del dataset

Esta plantilla asume el formato común de AG News con `train.csv` y `test.csv`.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df.head()


## 4. Exploración inicial

Revisa estructura, columnas, nulos y distribución de etiquetas.


In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nColumnas train:", train_df.columns.tolist())
print("\nInfo train:")
train_df.info()


In [ ]:
train_df.isnull().sum()


## 5. Preparación de texto y etiquetas

Ajusta los nombres de columnas si tu versión del dataset usa etiquetas distintas.


In [ ]:
TEXT_COL_1 = "Title"
TEXT_COL_2 = "Description"
LABEL_COL = "Class Index"

train_df["text"] = train_df[TEXT_COL_1].fillna("") + " " + train_df[TEXT_COL_2].fillna("")
test_df["text"] = test_df[TEXT_COL_1].fillna("") + " " + test_df[TEXT_COL_2].fillna("")

train_df["label"] = train_df[LABEL_COL]
test_df["label"] = test_df[LABEL_COL]

train_df[["label", "text"]].head()


In [ ]:
train_df["label"].value_counts().sort_index()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=train_df, x="label")
plt.title("Distribución de noticias por clase")
plt.savefig(FIGURES_DIR / "distribucion_clases.png", dpi=300)
plt.show()


In [ ]:
train_df["text_length"] = train_df["text"].str.len()
train_df["text_length"].describe()


## 6. Limpieza y normalización del texto

Revisa esta función y ajusta si decides conservar números, símbolos o stopwords.


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)


In [ ]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

train_df[["text", "clean_text"]].head()


## 7. Definición de entrenamiento y prueba

Si tu dataset ya viene separado, esta sección solo deja listas las variables. Si tienes un único archivo, sustituye esto por `train_test_split`.


In [ ]:
X_train = train_df["clean_text"]
y_train = train_df["label"]
X_test = test_df["clean_text"]
y_test = test_df["label"]

len(X_train), len(X_test)


## 8. Vectorización con TF-IDF

Ajusta los parámetros si quieres comparar configuraciones, pero documenta cualquier cambio.


In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

X_train_tfidf.shape, X_test_tfidf.shape


## 9. Entrenamiento de modelos supervisados

La plantilla incluye tres modelos recomendados para clasificación clásica de texto.


In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)


## 10. Evaluación de métricas

Reporta Accuracy, Precision, Recall y F1-score. Después interpreta los resultados en una celda Markdown.


In [ ]:
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted"
    )
    return {
        "modelo": name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

results = [
    evaluate_model("Naive Bayes", y_test, nb_pred),
    evaluate_model("Logistic Regression", y_test, lr_pred),
    evaluate_model("LinearSVC", y_test, svm_pred)
]

results_df = pd.DataFrame(results)
results_df


In [ ]:
results_df.to_csv(BASE_DIR / "metricas_modelos.csv", index=False)


## 11. Classification report del mejor modelo

Cambia `best_model_name` y `best_pred` si otro modelo obtiene mejor resultado.


In [ ]:
best_model_name = "LinearSVC"
best_pred = svm_pred

print(best_model_name)
print(classification_report(y_test, best_pred))


## 12. Matriz de confusión

Genera la matriz de confusión del modelo final y comenta qué clases se confunden más.


In [ ]:
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title(f"Matriz de confusión - {best_model_name}")
plt.savefig(FIGURES_DIR / f"matriz_confusion_{best_model_name.lower()}.png", dpi=300)
plt.show()


## 13. Validación cruzada

Usa pipelines para evitar fugas de información durante la vectorización.


In [ ]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3, max_df=0.95)),
    ("model", MultinomialNB())
])

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3, max_df=0.95)),
    ("model", LogisticRegression(max_iter=1000))
])

svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3, max_df=0.95)),
    ("model", LinearSVC())
])


In [ ]:
cv_nb = cross_val_score(nb_pipeline, X_train, y_train, cv=5, scoring="f1_weighted")
cv_lr = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring="f1_weighted")
cv_svm = cross_val_score(svm_pipeline, X_train, y_train, cv=5, scoring="f1_weighted")

cv_results = pd.DataFrame([
    {"modelo": "Naive Bayes", "cv_f1_mean": cv_nb.mean(), "cv_f1_std": cv_nb.std()},
    {"modelo": "Logistic Regression", "cv_f1_mean": cv_lr.mean(), "cv_f1_std": cv_lr.std()},
    {"modelo": "LinearSVC", "cv_f1_mean": cv_svm.mean(), "cv_f1_std": cv_svm.std()}
])
cv_results


## 14. Análisis de errores

Revisa ejemplos mal clasificados y comenta patrones observados.


In [ ]:
errors_df = pd.DataFrame({
    "text": X_test.values,
    "real": y_test.values,
    "predicho": best_pred
})

errors_df = errors_df[errors_df["real"] != errors_df["predicho"]]
errors_df.head(20)


In [ ]:
errors_df.to_csv(BASE_DIR / "ejemplos_errores.csv", index=False)


## 15. Comparación final y selección del modelo

Combina `results_df` y `cv_results`, luego justifica qué modelo eliges como final.


In [ ]:
final_comparison = results_df.merge(cv_results, on="modelo", how="left")
final_comparison


## 16. Conclusiones

Completa esta sección con:
- el mejor modelo,
- las métricas más relevantes,
- qué clases fueron más difíciles,
- limitaciones,
- mejoras futuras.
